In [ ]:
import numpy as np
import stim
from IPython.display import Markdown, display

from library.circuitry import Circuitry
from library.qubit_array import QubitArray
from library.deprecated.patch import SteaneCodePatch
from utils.fault_analysis import enumerate_faults

In [ ]:
# Implementation assisted by Claude.AI (October 2026)

In [ ]:
qubits: QubitArray = QubitArray(dimensions=(6, 6))
circuitry = Circuitry(qubits, clifford=True)
steane = SteaneCodePatch(array=qubits, anchor=(1, 1), injection=None)

steane.append_cultivation(circuitry)

circuitry.annotate_detector("STN:CULT:X0", "STN:CULT:X1")
for mk in range(2, 7):
    circuitry.annotate_detector(f"STN:CULT:X{mk}")
for mk in range(3):
    circuitry.annotate_detector(f"STN:CULT:Z{mk}")

circuitry.detectors_report(open_boundaries=True)

display(Markdown(f"[Open in Crumble (Cultivation)]({circuitry.to_crumble_url()})"))

In [ ]:
DATA = [0, 3, 4, 7, 10, 11, 12]
SUPPORTS = steane.stabilizers.values()
clifford = circuitry.as_stim()
clifford.diagram("timeline-svg")

In [ ]:
def flips_hxy(frame):
    _, zs = stim.PauliString(frame).to_numpy()
    return bool(zs[DATA].sum() % 2)


def detected_internally(flips):
    return np.frombuffer(flips, dtype=bool).any()


def detected_z_check(frame):
    xs, _ = stim.PauliString(frame).to_numpy()
    for support in SUPPORTS:
        if xs[support].sum() % 2 == 1:
            return True
    return False


def detected_x_check(frame):
    xs, zs = stim.PauliString(frame).to_numpy()

    for support in SUPPORTS:
        x, z = xs[support], zs[support]
        count_h = int(x.sum())
        if count_h == 0:
            if z.sum() % 2 == 1:
                return True
        elif count_h == 4:
            if (x & ~z).sum() % 2 == 1:
                return True
    return False

In [ ]:
faults, outcomes = enumerate_faults(circuitry)
group_count = len(outcomes)

print(f"Fault count [k=1] : {len(faults)}")
print(f"Group count [k=1] : {group_count}")

In [ ]:
fault_count = sum(
    len(configurations)
    for (frame, flips), configurations in outcomes.items()
    if flips_hxy(frame)
    and not detected_internally(flips)
    and not detected_z_check(frame)
    and not detected_x_check(frame)
)
group_count = sum(
    1
    for (frame, flips), configurations in outcomes.items()
    if flips_hxy(frame)
    and not detected_internally(flips)
    and not detected_z_check(frame)
    and not detected_x_check(frame)
)

print(
    f"Number of undetected Hxy-flips caused by single faults : {fault_count} [groups:{group_count}]"
)
for (frame, flips), configurations in outcomes.items():
    if (
        not flips_hxy(frame)
        or detected_internally(flips)
        or detected_z_check(frame)
        or detected_x_check(frame)
    ):
        continue

    print(f"Frame [{len(configurations)}] : {frame}")

    for op_idx, tk_idx, fault in configurations:
        pauli = "*".join(f"{p}{q}" for q, p in sorted(fault.items()))
        display(
            Markdown(
                f"[Open in Crumble (Cultivation {pauli}@T{tk_idx} : `{frame}`)]({circuitry.to_crumble_url(marks=[(op_idx, fault, 0)])})"
            )
        )

In [ ]:
faults, outcomes = enumerate_faults(circuitry, k=2)
group_count = len(outcomes)

print(f"Fault count [k=2] : {len(faults)}")
print(f"Group count [k=2] : {group_count}")

In [ ]:
fault_count = sum(
    len(configurations)
    for (frame, flips), configurations in outcomes.items()
    if flips_hxy(frame)
    and not detected_internally(flips)
    and not detected_z_check(frame)
    and not detected_x_check(frame)
)
group_count = sum(
    1
    for (frame, flips), configurations in outcomes.items()
    if flips_hxy(frame)
    and not detected_internally(flips)
    and not detected_z_check(frame)
    and not detected_x_check(frame)
)

print(
    f"Number of undetected Hxy-flips caused by double faults : {fault_count} [groups:{group_count}]"
)
for (frame, flips), configurations in outcomes.items():
    if (
        not flips_hxy(frame)
        or detected_internally(flips)
        or detected_z_check(frame)
        or detected_x_check(frame)
    ):
        continue

    print(f"Frame [{len(configurations)}] : {frame}")
    for configuration in configurations:
        paulis = []
        marks = []
        for ft_idx in configuration:
            op_idx, tk_idx, fault = faults[ft_idx]
            paulis.append(
                "*".join(f"{p}{q}" for q, p in sorted(fault.items())) + f"@T{tk_idx}"
            )
            marks.append((op_idx, fault, 0))
        display(
            Markdown(
                f"[Open in Crumble (Cultivation {' + '.join(paulis)})]({circuitry.to_crumble_url()})"
            )
        )